# Gradient Boosting - Day 3 (Scikit-Learn)

## Overfitting in Gradient Boosted Trees

Unlike random forests, gradient boosted trees can overfit. Therefore, as with neural networks, you can apply regularization and early stopping using a validation dataset.

---

## Signs of Overfitting

When training a GBT model, look for divergent curves:
- Training loss continues to decrease
- Validation loss decreases then starts increasing
- Training accuracy keeps improving
- Validation accuracy peaks then declines

---

## Common Regularization Parameters

| Parameter | Description | Effect |
|-----------|-------------|--------|
| `max_depth` | Maximum depth of each tree | Shallower trees = less overfitting |
| `learning_rate` | Shrinkage rate | Smaller values = more regularization |
| `max_features` | Ratio of attributes tested at each node | Less features = more diversity |
| `min_samples_split` | Minimum samples to split a node | Higher values = less overfitting |
| `min_samples_leaf` | Minimum samples in leaf | Higher values = smoother predictions |
| `subsample` | Fraction of samples to use per tree | Less samples = more randomness |

---

## L1 and L2 Regularization

Scikit-Learn's Gradient Boosting includes regularization parameters:
- `alpha`: L1 regularization on leaf weights (for quantile/huber loss)
- No direct L2 on leaf weights, but `min_samples_split` and `min_samples_leaf` provide similar effects

---

## Early Stopping

Early stopping stops training when validation performance stops improving.

**Key parameters:**
- `validation_fraction`: Fraction of training data to use as validation
- `n_iter_no_change`: Number of iterations with no improvement before stopping
- `tol`: Tolerance for improvement

---

## Tree Depth in GBT vs Random Forest

| Model | Typical Depth | Reason |
|-------|---------------|--------|
| Random Forest | Deeper (10-100) | Bagging reduces variance |
| Gradient Boosting | Shallower (3-6) | Sequential training can overfit |

---

## Pros and Cons of Gradient Boosted Trees

**Pros:**
- Native support for numerical and categorical features
- Default hyperparameters often give great results
- Small model size and fast inference (µs per example)

**Cons:**
- Must train trees sequentially (slower training)
- Cannot learn reusable internal representations (unlike neural networks)
- Poorer results on unstructured data (images, text, audio)

---

## One-Line Summary

**Gradient boosted trees can overfit, so use regularization parameters and early stopping with a validation dataset.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

In [ ]:
# Create dataset
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## Overfitting Demonstration

In [ ]:
# Train with many trees (no early stopping)
n_estimators_range = [1, 10, 20, 50, 100, 200, 300, 400, 500]
train_scores = []
test_scores = []

for n in n_estimators_range:
    gb = GradientBoostingClassifier(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, gb.predict(X_train)))
    test_scores.append(accuracy_score(y_test, gb.predict(X_test)))

print("Training completed")

In [ ]:
# Plot overfitting curve
plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, train_scores, 'b-', linewidth=2, label='Training Accuracy')
plt.plot(n_estimators_range, test_scores, 'r--', linewidth=2, label='Test Accuracy')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Overfitting in Gradient Boosted Trees')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\nObservation:")
print("- Training accuracy keeps increasing")
print("- Test accuracy peaks then decreases -> OVERFITTING")

## Early Stopping

In [ ]:
# Gradient Boosting with early stopping
gb_early = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=3,
    validation_fraction=0.1,
    n_iter_no_change=10,
    tol=1e-4,
    random_state=42
)

gb_early.fit(X_train, y_train)

print(f"Actual number of trees used: {gb_early.n_estimators_}")
print(f"Test accuracy: {accuracy_score(y_test, gb_early.predict(X_test)):.4f}")

## Regularization Parameters Effect

In [ ]:
# Effect of max_depth
print("\n" + "="*50)
print("Effect of Max Depth")
print("="*50)

depths = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
depth_scores = []

for d in depths:
    gb = GradientBoostingClassifier(n_estimators=100, max_depth=d, random_state=42)
    gb.fit(X_train, y_train)
    acc = accuracy_score(y_test, gb.predict(X_test))
    depth_scores.append(acc)
    print(f"max_depth={d:2}: Test Accuracy = {acc:.4f}")

In [ ]:
# Effect of learning rate
print("\n" + "="*50)
print("Effect of Learning Rate (Shrinkage)")
print("="*50)

learning_rates = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
lr_scores = []

for lr in learning_rates:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=42)
    gb.fit(X_train, y_train)
    acc = accuracy_score(y_test, gb.predict(X_test))
    lr_scores.append(acc)
    print(f"learning_rate={lr:.2f}: Test Accuracy = {acc:.4f}")

In [ ]:
# Effect of subsample
print("\n" + "="*50)
print("Effect of Subsample (Stochastic Gradient Boosting)")
print("="*50)

subsamples = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for ss in subsamples:
    gb = GradientBoostingClassifier(n_estimators=100, subsample=ss, random_state=42)
    gb.fit(X_train, y_train)
    acc = accuracy_score(y_test, gb.predict(X_test))
    print(f"subsample={ss:.1f}: Test Accuracy = {acc:.4f}")

In [ ]:
# Effect of min_samples_split
print("\n" + "="*50)
print("Effect of Min Samples Split")
print("="*50)

min_samples = [2, 5, 10, 20, 50, 100]

for ms in min_samples:
    gb = GradientBoostingClassifier(n_estimators=100, min_samples_split=ms, random_state=42)
    gb.fit(X_train, y_train)
    acc = accuracy_score(y_test, gb.predict(X_test))
    print(f"min_samples_split={ms:3}: Test Accuracy = {acc:.4f}")

## Best Regularized Model

In [ ]:
# Best regularized model
gb_best = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    min_samples_split=10,
    min_samples_leaf=5,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

gb_best.fit(X_train, y_train)

print("Best Regularized Model:")
print("="*40)
print(f"Trees used: {gb_best.n_estimators_}")
print(f"Train Accuracy: {accuracy_score(y_train, gb_best.predict(X_train)):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, gb_best.predict(X_test)):.4f}")

In [ ]:
# Compare models
print("\n" + "="*50)
print("Model Comparison")
print("="*50)

models = {
    'No Regularization': GradientBoostingClassifier(n_estimators=500, random_state=42),
    'With Early Stopping': GradientBoostingClassifier(n_estimators=500, validation_fraction=0.1, n_iter_no_change=10, random_state=42),
    'Regularized': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"\n{name}:")
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Gap: {train_acc - test_acc:.4f}")

In [ ]:
# Day 3 Completed
print("\n" + "="*40)
print("DAY 3 COMPLETED")
print("="*40)
print("Topics covered:")
print("- Overfitting in Gradient Boosted Trees")
print("- Signs of overfitting (diverging curves)")
print("- Regularization parameters (max_depth, learning_rate, subsample)")
print("- Early stopping with validation data")
print("- Pros and Cons of GBT")
print("- Parameter effect demonstrations")
print("="*40)"
print("\n*** GRADIENT BOOSTING MODULE COMPLETED ***")
print("\nNext: XGBoost from Scratch")